# 이슈 #38 구조환경지표 원자료 기반 전수검증

## 목적과 현재 범위

- `configs/structural_indicators_verification.yaml`을 기준으로 구조환경지표 21개의 가공값과 원자료 연결 상태를 점검한다.
- 현재 상세 검증이 완료된 지표는 `2-1.1. 보육시설 보급률`이다. 나머지 지표의 산식 검증은 후속 작업이다.
- 기존 가공값은 `어린이집 수 ÷ 0–4세 인구 × 100`과 전부 일치했으며, 명세상 올바른 산식은 `어린이집 정원 ÷ 0–4세 인구 × 100`이다.
- 이 노트북은 수정값을 메모리에 생성할 뿐 원본 또는 기존 가공 파일을 덮어쓰지 않는다.

## 1. 실행 환경과 검증 명세 확인

저장소 루트를 찾고 필수 라이브러리를 불러온 뒤, 검증 명세와 구조환경 원자료 경로가 존재하는지 확인한다. YAML의 필수 구역, 지표 수, 지표 ID 중복 여부도 검사한다. 정상 출력은 이슈 `#38`, 검증 대상 `21개`, 고유 ID `21개`이다.

In [1]:
import hashlib
import unicodedata

import openpyxl
import pandas as pd
import yaml

from pathlib import Path

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()), None
)

if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")

print(f"저장소 루트: {repo_root}")

저장소 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


In [2]:
config_path = repo_root / "configs" / "structural_indicators_verification.yaml"
raw_root = repo_root / "data" / "raw"

missing_paths = [path for path in (config_path, raw_root) if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "다음 입력 경로를 찾지 못했습니다:\n" + "\n".join(map(str, missing_paths))
    )

print(f"검증 명세: {config_path.relative_to(repo_root)}")
print(f"구조환경 원자료: {raw_root.relative_to(repo_root)}")

검증 명세: configs/structural_indicators_verification.yaml
구조환경 원자료: data/raw


In [3]:
with config_path.open(encoding="utf-8") as file:
    manifest = yaml.safe_load(file)

required_sections = {"manifest_version", "issue", "scope", "rules", "local", "drive", "indicators"}
missing_sections = required_sections - manifest.keys()
if missing_sections:
    raise KeyError(f"YAML 필수 구역이 없습니다: {sorted(missing_sections)}")

indicators = manifest["indicators"]
if not isinstance(indicators, list) or not all(
    isinstance(indicator, dict) for indicator in indicators
):
    raise TypeError("indicators는 지표 딕셔너리로 구성된 리스트여야 합니다.")

indicator_ids = [indicator.get("id") for indicator in indicators]
if len(indicators) != manifest["scope"]["completed_indicators_to_validate"]:
    raise ValueError("검증 대상 지표 수가 scope의 선언값과 다릅니다.")
if None in indicator_ids or len(indicator_ids) != len(set(indicator_ids)):
    raise ValueError("지표 ID가 없거나 중복됐습니다.")

print(
    f"명세 버전: {manifest['manifest_version']} | 이슈: #{manifest['issue']} | "
    f"검증 대상: {len(indicators)}개 | 고유 ID: {len(set(indicator_ids))}개"
)

명세 버전: 2 | 이슈: #38 | 검증 대상: 21개 | 고유 ID: 21개


## 2. 명세와 로컬 원자료 연결 확인

YAML에 기록된 가공 파일·원자료 파일명을 로컬 원자료 폴더와 연결한다. 같은 이름의 파일이 여러 개면 SHA-256 해시로 실제 내용까지 비교한다. 현재 누락 파일은 없지만, 주민등록인구 파일은 같은 이름의 서로 다른 내용이 있어 지표에 맞는 파일을 명시적으로 선택해야 한다.

In [ ]:
def normalize_name(name: str) -> str:
    return unicodedata.normalize("NFC", name)


local_files: list[Path] = sorted(
    (path for path in raw_root.rglob("*") if path.is_file()),
    key=lambda path: path.as_posix().casefold(),
)

if not local_files:
    raise FileNotFoundError(f"원자료 폴더가 비어 있습니다: {raw_root}")

paths_by_name: dict[str, list[Path]] = {}

for path in local_files:
    paths_by_name.setdefault(normalize_name(path.name), []).append(path)

mapping_records: list[dict[str, str]] = []

for indicator in indicators:
    mapping_records.append(
        {
            "indicator_id": indicator["id"],
            "mapping_type": "finance_team_value",
            "role": "finance_team_value",
            "drive_id": indicator["finance_team_value"]["id"],
            "file_name": indicator["finance_team_value"]["file_name"],
        }
    )

    for source in indicator["raw_sources"]:
        mapping_records.append(
            {
                "indicator_id": indicator["id"],
                "mapping_type": "raw",
                "role": source["role"],
                "drive_id": source["id"],
                "file_name": source["file_name"],
            }
        )

resolution_rows: list[dict[str, object]] = []

for record in mapping_records:
    matched_paths = paths_by_name.get(normalize_name(record["file_name"]), [])

    resolution_rows.append(
        {
            **record,
            "match_count": len(matched_paths),
            "local_paths": " | ".join(str(path.relative_to(repo_root)) for path in matched_paths),
        }
    )

file_resolution = pd.DataFrame(resolution_rows)

missing_file_mappings = file_resolution[file_resolution["match_count"] == 0].copy()

ambiguous_file_mappings = file_resolution[file_resolution["match_count"] > 1].copy()

print(
    f"로컬 탐색 파일: {len(local_files)}개 | "
    f"매핑 참조: {len(file_resolution)}건 | "
    f"정확히 1개 일치: {(file_resolution['match_count'] == 1).sum()}건 | "
    f"누락: {len(missing_file_mappings)}건 | "
    f"다중 일치: {len(ambiguous_file_mappings)}건"
)

with pd.option_context("display.max_colwidth", None):
    if not missing_file_mappings.empty:
        print(f"\n누락된 매핑 ({len(missing_file_mappings)}건):")
        display(
            missing_file_mappings[["indicator_id", "role", "file_name"]]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

    if not ambiguous_file_mappings.empty:
        print(f"\n다중 일치 매핑 ({len(ambiguous_file_mappings)}건):")
        display(
            ambiguous_file_mappings[
                ["indicator_id", "role", "file_name", "match_count", "local_paths"]
            ]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

In [5]:
def calculate_sha256(path: Path) -> str:
    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


ambiguous_names = sorted(
    str(file_name) for file_name in ambiguous_file_mappings["file_name"].unique()
)

hash_rows: list[dict[str, object]] = []

for file_name in ambiguous_names:
    for path in paths_by_name[normalize_name(file_name)]:
        hash_rows.append(
            {
                "file_name": file_name,
                "local_path": path.relative_to(raw_root).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": calculate_sha256(path),
            }
        )

duplicate_file_hashes = pd.DataFrame(hash_rows)

duplicate_content_summary = duplicate_file_hashes.groupby("file_name", as_index=False).agg(
    candidate_count=("local_path", "size"),
    distinct_size_count=("size_bytes", "nunique"),
    distinct_hash_count=("sha256", "nunique"),
)

duplicate_content_summary["content_identical"] = (
    duplicate_content_summary["distinct_hash_count"] == 1
)

different_content_files = duplicate_content_summary[
    ~duplicate_content_summary["content_identical"]
].copy()

print(
    f"중복 파일명: {len(duplicate_content_summary)}개 | "
    f"후보 파일: {len(duplicate_file_hashes)}개 | "
    f"내용 동일: {duplicate_content_summary['content_identical'].sum()}개 | "
    f"내용 다름: {len(different_content_files)}개"
)

if not different_content_files.empty:
    print("\n내용이 다른 중복 파일:")
    display(different_content_files)
    display(
        duplicate_file_hashes[
            duplicate_file_hashes["file_name"].isin(different_content_files["file_name"])
        ]
    )

KeyError: 'file_name'

## 3. 보육시설 보급률 입력 파일 확정

- `finance_team_value`: 검증할 기존 보육시설 보급률
- `numerator_active`: 올바른 분자인 어린이집 정원
- `denominator`: 0–4세 주민등록인구
- `numerator_excluded_facility_count`: 기존 오류 산식을 입증하기 위한 어린이집 수 비교자료이며, 수정 산식의 분자로 사용하지 않는다.

내용이 다른 동명 주민등록인구 파일 가운데 `2-1.1. 보육시설 보급률 원자료` 폴더의 파일을 분모로 고정한다.

In [ ]:
# cell 8
population_path = raw_root / (
    "3. 제주여성가족연구원 지표별 측정값(raw)/"
    "2-1.1. 보육시설 보급률 원자료/"
    "2016-2025_행정구역(시도)별_1세별_주민등록인구_20260715.csv.xlsx"
)

if not population_path.is_file():
    raise FileNotFoundError(population_path)

local_path_overrides = {("childcare_capacity_rate", "raw", "denominator"): population_path}

In [ ]:
# cell 9

childcare = next(item for item in indicators if item["id"] == "childcare_capacity_rate")

childcare_paths = {
    "finance_team_value": paths_by_name[childcare["finance_team_value"]["file_name"]][0],
    **{
        source["role"]: local_path_overrides.get(
            (childcare["id"], "raw", source["role"]),
            paths_by_name[source["file_name"]][0],
        )
        for source in childcare["raw_sources"]
    },
}

for role, path in childcare_paths.items():
    print(f"{role}: {path.relative_to(repo_root)}")

## 4. 입력 자료 구조 확인

선택한 엑셀 파일의 시트명, 행·열 크기, 상단 값을 미리 확인한다. 이 단계에서는 값을 변환하지 않으며, 2016–2025년 지역별 정원과 0–4세 인구가 실제로 들어 있는지 확인한다.

In [ ]:
# cell 10

for role, path in childcare_paths.items():
    if path.suffix.lower() != ".xlsx":
        continue

    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    print(f"\n[{role}] {path.name}")

    for worksheet in workbook.worksheets:
        preview = list(worksheet.iter_rows(max_row=8, values_only=True))
        print(f"{worksheet.title}: {worksheet.max_row}행 × {worksheet.max_column}열")
        display(pd.DataFrame(preview))

    workbook.close()

### 비교용 어린이집 수 자료

기존 가공값이 어린이집 수를 사용했는지 확인하기 위해 CSV를 별도로 읽는다. 인코딩을 자동 판별하며, 현재 파일은 `cp949`, 18개 지역 × 10개 연도 구조이다.

In [ ]:
# cell 11

facility_count_path = childcare_paths["numerator_excluded_facility_count"]

for encoding in ("utf-8-sig", "cp949", "euc-kr"):
    try:
        facility_count_raw = pd.read_csv(
            facility_count_path,
            encoding=encoding,
        )
        facility_count_encoding = encoding
        break
    except UnicodeDecodeError:
        continue
else:
    raise UnicodeError(f"CSV 인코딩을 확인하지 못했습니다: {facility_count_path}")

print(f"인코딩: {facility_count_encoding}")
print(f"크기: {facility_count_raw.shape[0]}행 × {facility_count_raw.shape[1]}열")
print(f"열 이름: {facility_count_raw.columns.tolist()}")

display(facility_count_raw.head(8))

## 5. 정원·시설 수·0–4세 인구 결합

지역명을 `서울특별시 → 서울`처럼 통일하고 세 자료를 `지역 × 연도` 긴 형식으로 변환한다. 주민등록인구는 0–4세의 `총인구수[명]`만 합산한다. 이후 세 자료를 일대일 결합하고, 18개 지역 × 10개 연도인 180행과 결측값 없음까지 검증한다. 결과 변수는 `childcare_comparison`이다.

In [ ]:
# cell 12

years = [str(y) for y in range(2016, 2026)]
region_order = [
    "전국",
    "서울",
    "부산",
    "대구",
    "인천",
    "광주",
    "대전",
    "울산",
    "세종",
    "경기",
    "강원",
    "충북",
    "충남",
    "전북",
    "전남",
    "경북",
    "경남",
    "제주",
]

region_map = {r: r for r in region_order} | {
    "서울특별시": "서울",
    "부산광역시": "부산",
    "대구광역시": "대구",
    "인천광역시": "인천",
    "광주광역시": "광주",
    "대전광역시": "대전",
    "울산광역시": "울산",
    "세종특별자치시": "세종",
    "경기도": "경기",
    "강원도": "강원",
    "강원특별자치도": "강원",
    "충청북도": "충북",
    "충청남도": "충남",
    "전라북도": "전북",
    "전북특별자치도": "전북",
    "전라남도": "전남",
    "경상북도": "경북",
    "경상남도": "경남",
    "제주특별자치도": "제주",
}


def wide_to_long(df, region_col, value_col):
    df = df.copy()
    df.columns = df.columns.map(lambda x: str(x).strip())
    df["지역"] = df[region_col].astype(str).str.strip().map(region_map)

    if df["지역"].isna().any():
        raise ValueError(f"{value_col}: 미등록 지역명이 있습니다.")
    if set(df["지역"]) != set(region_order) or df["지역"].duplicated().any():
        raise ValueError(f"{value_col}: 지역 누락 또는 중복이 있습니다.")

    result = df[["지역", *years]].melt(id_vars="지역", var_name="연도", value_name=value_col)
    result["연도"] = result["연도"].astype(int)
    result[value_col] = pd.to_numeric(result[value_col], errors="raise")
    return result


capacity_raw = pd.read_excel(childcare_paths["numerator_active"])

population_raw = pd.read_excel(childcare_paths["denominator"])
population_raw.columns = population_raw.columns.map(lambda x: str(x).strip())
population_raw = population_raw.rename(columns={f"{y} 년": y for y in years})

for col in ["행정구역(시군구)별", "연령별", "항목"]:
    population_raw[col] = population_raw[col].astype(str).str.strip()

population_selected = population_raw[
    population_raw["행정구역(시군구)별"].isin(region_map)
    & population_raw["연령별"].isin([f"{age}세" for age in range(5)])
    & population_raw["항목"].eq("총인구수[명]")
].copy()

population_selected["지역"] = population_selected["행정구역(시군구)별"].map(region_map)

age_counts = population_selected.groupby("지역").size().reindex(region_order, fill_value=0)
if not age_counts.eq(5).all():
    raise ValueError(f"지역별 0–4세 자료 오류:\n{age_counts[age_counts.ne(5)]}")

population_selected[years] = population_selected[years].apply(pd.to_numeric, errors="raise")
population_wide = population_selected.groupby("지역", as_index=False)[years].sum()

childcare_comparison = (
    wide_to_long(capacity_raw, "시도별", "어린이집_정원")
    .merge(
        wide_to_long(facility_count_raw, "시도별", "어린이집_수"),
        on=["지역", "연도"],
        validate="one_to_one",
    )
    .merge(
        wide_to_long(population_wide, "지역", "0_4세_인구"),
        on=["지역", "연도"],
        validate="one_to_one",
    )
)

order_map = {r: i for i, r in enumerate(region_order)}
childcare_comparison["_순서"] = childcare_comparison["지역"].map(order_map)
childcare_comparison = (
    childcare_comparison.sort_values(["_순서", "연도"]).drop(columns="_순서").reset_index(drop=True)
)

if len(childcare_comparison) != 180 or childcare_comparison.isna().any().any():
    raise ValueError("결합 결과에 행 수 오류 또는 결측값이 있습니다.")

print(f"결합 결과: {childcare_comparison.shape}")
display(childcare_comparison.head(12))

## 6. 기존 산식 오류 검증

두 산식을 같은 지역·연도 자료로 계산한다.

- 정원 기준: `어린이집 정원 ÷ 0–4세 인구 × 100`
- 개수 기준: `어린이집 수 ÷ 0–4세 인구 × 100`

기존 가공값과 개수 기준 값의 절대오차를 계산한다. `기존값 최대 오차: 0.0`은 180개 지역·연도 값 모두가 개수 기준 산식과 정확히 일치한다는 뜻이며, 기존 산식 오류가 확정된다.

In [ ]:
# cell 13

childcare_comparison["정원_기준_보급률"] = (
    childcare_comparison["어린이집_정원"] / childcare_comparison["0_4세_인구"] * 100
)
childcare_comparison["개수_기준_보급률"] = (
    childcare_comparison["어린이집_수"] / childcare_comparison["0_4세_인구"] * 100
)

existing_raw = pd.read_excel(
    childcare_paths["finance_team_value"], sheet_name="2-1.1. 보육시설 보급률"
)
existing_rate = wide_to_long(existing_raw, "지역", "기존_보급률")

childcare_comparison = childcare_comparison.merge(
    existing_rate, on=["지역", "연도"], validate="one_to_one"
)
childcare_comparison["기존값_오차"] = (
    childcare_comparison["기존_보급률"] - childcare_comparison["개수_기준_보급률"]
).abs()

print("기존값 최대 오차:", childcare_comparison["기존값_오차"].max())
display(childcare_comparison.head(12).round(6))

## 7. 수정값 표 생성과 인계 상태

`정원_기준_보급률`을 기존 가공 파일과 같은 `지역 × 연도` 형태로 변환해 `childcare_corrected`에 저장한다. 정상 결과는 18행 × 11열이며 결측값이 없어야 한다.

**현재 완료:** 보육시설 보급률의 원자료 연결, 기존 오류 입증, 2016–2025년 수정값 생성.

**후속 작업:** 수정값을 실제 가공 파일·통합본에 반영하고 일치 여부를 재검증한 뒤, 나머지 구조환경지표의 산식 검증을 이어간다.

In [ ]:
# cell 14: 수정된 보육시설 보급률 표 생성

childcare_corrected = (
    childcare_comparison.pivot(index="지역", columns="연도", values="정원_기준_보급률")
    .reindex(index=region_order, columns=range(2016, 2026))
    .reset_index()
)

if childcare_corrected.shape != (18, 11) or childcare_corrected.isna().any().any():
    raise ValueError("수정 결과의 지역·연도 또는 결측값을 확인하세요.")

display(childcare_corrected.round(6))